In [9]:
import wandb
import os

os.environ['WANDB_NOTEBOOK_NAME'] = 'sweeps.ipynb'

wandb.login()


True

# 1 - Sweep Definition
First we define the sweep with all the method, parameters and strategies

## Grid Search Configuration

In [10]:
sweep_config = {
    "method": "random"
}

#Metric is used just on bayesian sweeps, is the strategy used by it
metric = {
    "name": "loss",
    "goal": "minimize"
}

sweep_config["metric"] = metric

#Dictionary with the parameters to be tested
parameters_dict = {
    'optimizer': {
        'values': ['adam']
    },
    'batch_size': {
        'values': [32]
    },
     'fc_layer_size': {
        'values': [128]
        },
    'dropout': {
          'values': [0.3, 0.4, 0.5]
        }
}

sweep_config["parameters"] = parameters_dict

#To set a parameter that we dont want to optimize and just set as it is
parameters_dict.update({
    'epochs': {
        'value': 5
    }
})


# Random Search Configuration
We specificy the range of parameters in which the random search will operate. \
To see all the distribution types: https://docs.wandb.ai/guides/sweeps/define-sweep-configuration#distributions

In [11]:
parameters_dict.update({
    'learning_rate': {
        # a flat distribution between 0 and 0.1, check the documentation of all the distributions types
        'distribution': 'uniform',
        'min': 0,
        'max': 0.1
      },
    'batch_size': {
        # integers between 32 and 256
        # with evenly-distributed logarithms 
        'distribution': 'q_log_uniform_values',
        'q': 8,
        'min': 32,
        'max': 256,
      }
    })

In [12]:
import pprint

#Print of the nested dictaionary with the configuration of the sweep
pprint.pprint(sweep_config)

{'method': 'random',
 'metric': {'goal': 'minimize', 'name': 'loss'},
 'parameters': {'batch_size': {'distribution': 'q_log_uniform_values',
                               'max': 256,
                               'min': 32,
                               'q': 8},
                'dropout': {'values': [0.3, 0.4, 0.5]},
                'epochs': {'value': 5},
                'fc_layer_size': {'values': [128]},
                'learning_rate': {'distribution': 'uniform',
                                  'max': 0.1,
                                  'min': 0},
                'optimizer': {'values': ['adam']}}}


# 2 - Sweep controller initialization
This command below initializes a sweep on a remote machine with the parameters that we specified, ready for some agent (machines) to ask for a task to do. The remote machine will send the task to the agent containing a set of values for the parameters to try.

In [13]:
#Creation of the sweep, for now we have created a sweep with the ID below
#m7d3g2ox
sweep_id = wandb.sweep(sweep_config, project="EXIST2024")


# 3 - Defining a demo NN in Pytorch
This is just a demo NN where we insert the wandb components
wandb.init: Initialize a new W&B Run. Each Run is a single execution of the training function. \
wandb.config: Save all your hyperparameters in a configuration object so they can be logged. Read more about how to use wandb.config \
wandb.log: Log model behavior to W&B. Here, we just log the performance

In [14]:
import torch
import torch.optim as optim
import torch.nn.functional as F
import torch.nn as nn
from torchvision import datasets, transforms
import tqdm as notebook_tqdm


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def build_dataset(batch_size):
   
    transform = transforms.Compose(
        [transforms.ToTensor(),
         transforms.Normalize((0.1307,), (0.3081,))])
    # download MNIST training dataset

   

    dataset = datasets.MNIST("../data", train=True, download=True,
                             transform=transform)
    sub_dataset = torch.utils.data.Subset(
        dataset, indices=range(0, len(dataset), 5))
    loader = torch.utils.data.DataLoader(sub_dataset, batch_size=batch_size)

    return loader

def build_network(fc_layer_size, dropout):
    network = nn.Sequential(  # fully-connected, single hidden layer
        nn.Flatten(),
        nn.Linear(784, fc_layer_size), nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(fc_layer_size, 10),
        nn.LogSoftmax(dim=1))

    return network.to(device)
        

def build_optimizer(network, optimizer, learning_rate):
    if optimizer == "sgd":
        optimizer = optim.SGD(network.parameters(),
                              lr=learning_rate, momentum=0.9)
    elif optimizer == "adam":
        optimizer = optim.Adam(network.parameters(),
                               lr=learning_rate)
    return optimizer


def train_epoch(network, loader, optimizer):
    cumu_loss = 0
    for _, (data, target) in enumerate(loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()

        # ➡ Forward pass
        loss = F.nll_loss(network(data), target)
        cumu_loss += loss.item()

        # ⬅ Backward pass + weight update
        loss.backward()
        optimizer.step()

        wandb.log({"batch loss": loss.item()})

    return cumu_loss / len(loader)

def train(config=None):
    # Initialize a new wandb run
    with wandb.init(config=config):
        # If called by wandb.agent, as below,
        # this config will be set by Sweep Controller
        config = wandb.config

        loader = build_dataset(config.batch_size)
        network = build_network(config.fc_layer_size, config.dropout)
        optimizer = build_optimizer(network, config.optimizer, config.learning_rate)

        for epoch in range(config.epochs):
            avg_loss = train_epoch(network, loader, optimizer)
            wandb.log({"loss": avg_loss, "epoch": epoch})           

# 4 - Running the sweep with an agent
sweep_id: The sweeeps id created when we execute the controller \
train: the training function to use in the sweep \
count: the number of times that executes the training, is needed for the random search. In case of grid search the algorithm finishes when we finish the combinations of parameters

In [15]:
wandb.agent("/EXIST2024/wsfphj24", train, count=1)

wandb: Agent Starting Run: zozpsvjx with config:
wandb: 	batch_size: 168
wandb: 	dropout: 0.5
wandb: 	epochs: 5
wandb: 	fc_layer_size: 128
wandb: 	learning_rate: 0.00928759952222541
wandb: 	optimizer: adam


batch loss,█▃▃▂▂▁▂▂▂▁▂▂▂▁▂▁▂▁▁▂▁▁▁▁▂▂▁▂▁▁▂▁▁▂▁▂▁▂▂▂
epoch,▁▃▅▆█
loss,█▃▁▁▁
batch loss,0.4165
epoch,4
loss,0.38128


In [ ]:
'''wandb.run.summary["acc_training_best"] = best_training_accuracy
wandb.run.summary["loss_training_best"] = best_training_loss

wandb.run.summary["loss_validation_best"] = best_validation_loss
wandb.run.summary["acc_validation_best"] = best_validation_accuracy'''